In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os


In [ ]:

# ==========================================
# 0. 参数设置与读取数据
# ==========================================
# 请确保这里的文件路径和之前保存的一致
file_path = 'results/adata_macrophage_with_func_scores.h5ad' 
cluster_col = 'cell_type2'  # 如果您的列名是 'Final_Cell_Type'，请修改这里
time_col = 'time_point'    # 时间点列名

print(f"正在读取数据: {file_path} ...")
adata_mp = sc.read_h5ad(file_path)
print(f"读取成功！包含 {adata_mp.n_obs} 个细胞，{adata_mp.n_vars} 个基因。")

# 确保时间点是有序的分类变量（对于后面画小提琴图很重要）
time_order = ['D0', 'D10', 'D14', 'D33', 'D49', 'D61']
if adata_mp.obs[time_col].dtype.name != 'category' or list(adata_mp.obs[time_col].cat.categories) != time_order:
    adata_mp.obs[time_col] = pd.Categorical(adata_mp.obs[time_col], categories=time_order, ordered=True)

# 统一一下配色，保持论文一致性
custom_colors = {'FCN1+MP': '#0f4a86', 'MARCO+MP': '#4095d6', 'MP1': '#f9c2d5'}
adata_mp.uns[f'{cluster_col}_colors'] = [custom_colors[c] for c in adata_mp.obs[cluster_col].cat.categories]

os.makedirs('results/trajectory', exist_ok=True)


In [ ]:

# ==========================================
# 1. 计算 PAGA (发育轨迹与连通性图)
# ==========================================
print("正在计算细胞相邻图 (Neighbors) 与 PAGA 拓扑结构...")
# 重新计算一次邻接图以确保 PAGA 运行无误 (使用降维后的 PCA)
sc.pp.neighbors(adata_mp, n_neighbors=15, use_rep='X_pca')

# 运行 PAGA 分区图抽象算法
sc.tl.paga(adata_mp, groups=cluster_col)



In [ ]:
# ==========================================
# 2. 计算 DPT (扩散拟时序)
# ==========================================
print("正在推断拟时序 (Pseudotime)...")
# 我们需要定义一个“起点 (Root)”。
# 从生物学角度看，FCN1 通常是外周血单核细胞（早期/急性期进入组织），所以把它设为起点
root_target_cluster = 'FCN1+MP'
root_mask = adata_mp.obs[cluster_col] == root_target_cluster

# 选择该 cluster 中第一个细胞作为发育树的根节点
adata_mp.uns['iroot'] = np.flatnonzero(root_mask)[0] 

# 运行 DPT
sc.tl.dpt(adata_mp)


In [ ]:
# ==========================================
# 3. 绘图 - 1: PAGA 拓扑图与 UMAP 映射 (去除不兼容参数)
# ==========================================
print("正在绘制轨迹与拟时序图表...")
fig, axs = plt.subplots(1, 2, figsize=(14, 6), dpi=150)

# 左图：单纯的 PAGA 拓扑网络图
sc.pl.paga(
    adata_mp, 
    color=cluster_col, 
    title="PAGA Topology Graph", 
    ax=axs[0], 
    show=False,
    fontsize=12,
    node_size_scale=2
)

# 右图：在 UMAP 上叠加 PAGA 连线
# 3.1 先画底层的 UMAP 散点 (调低透明度 alpha)
sc.pl.umap(
    adata_mp, 
    color=cluster_col, 
    title="Trajectory on UMAP",
    ax=axs[1], 
    show=False,
    alpha=0.3
)

# 3.2 计算每个细胞亚群在 UMAP 上的中心坐标
categories = adata_mp.obs[cluster_col].cat.categories
umap_pos = np.zeros((len(categories), 2))
for i, cat in enumerate(categories):
    umap_pos[i] = adata_mp.obsm['X_umap'][adata_mp.obs[cluster_col] == cat].mean(axis=0)
    
# 3.3 将 PAGA 的连线直接画在这些中心点上
sc.pl.paga(
    adata_mp, 
    pos=umap_pos,         
    color=cluster_col,
    ax=axs[1], 
    show=False,
    node_size_scale=1.5  # 删除了导致报错的 edge_cmap_scale 参数
)

plt.tight_layout()
plt.savefig('results/trajectory/Macrophage_PAGA_Topology.pdf', bbox_inches='tight')
plt.show()

# ==========================================
# 4. 绘图 - 2: UMAP 上的拟时序 (颜色渐变)
# ==========================================
sc.pl.umap(
    adata_mp, 
    color='dpt_pseudotime', 
    color_map='viridis', # 经典拟时序配色
    title="Pseudotime (Root: FCN1+MP)", 
    save="_macrophage_pseudotime.pdf"
)

# ==========================================
# 5. 绘图 - 3: 拟时序 vs 实际实验时间点
# ==========================================
plt.figure(figsize=(8, 5), dpi=150)
import seaborn as sns

sns.violinplot(
    data=adata_mp.obs, 
    x=time_col, 
    y='dpt_pseudotime', 
    palette='Set3',
    inner='quartile', 
    linewidth=1.5
)
plt.title('Pseudotime Progression Across Real Time Points', fontsize=14, fontweight='bold')
plt.ylabel('Inferred Pseudotime', fontsize=12)
plt.xlabel('Actual Time Point', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('results/trajectory/Pseudotime_vs_Realtime_Violin.pdf', bbox_inches='tight')
plt.show()

print("✅ 所有轨迹分析与绘图已完成！")


In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 调整图片的大小，使其更紧凑美观
plt.rcParams['figure.figsize'] = (5, 4)

# 绘制与参考图一致的 UMAP 拟时序图
sc.pl.umap(
    adata_mp, 
    color='dpt_pseudotime', 
    color_map='magma',      # 关键参数：'magma' 完美对应您图中的黑-紫-橙-黄渐变
    title='Pseudotime',
    size=30,                # 散点的大小，如果您觉得点太稀疏或太挤，可以调大或调小这个数值 (例如 20 或 50)
    frameon=False,          # 设为 False 会去掉外部的方框，更加清爽，符合顶级期刊审美
    show=False
)

# 调整布局并高分辨率保存
plt.tight_layout()
plt.savefig('results/trajectory/UMAP_Pseudotime_magma.pdf', dpi=300, bbox_inches='tight')
plt.show()

# 恢复默认画幅大小，以免影响后续其他画图
plt.rcParams['figure.figsize'] = (4, 4)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

# 1. 基础准备与数据提取
cluster_col = 'cell_type2'  # 如果您的列名不同请修改
categories = adata_mp.obs[cluster_col].cat.categories
colors = adata_mp.uns[f'{cluster_col}_colors']
color_dict = dict(zip(categories, colors))

# 提取 UMAP 坐标与每个 cluster 的中心坐标
umap_coords = adata_mp.obsm['X_umap']
centers = {}
for cat in categories:
    mask = adata_mp.obs[cluster_col] == cat
    centers[cat] = umap_coords[mask].mean(axis=0)

# 提取 PAGA 连通性矩阵 (权重矩阵)
paga_conn = adata_mp.uns['paga']['connectivities'].toarray()

# 2. 开始构建画布
fig, ax = plt.subplots(figsize=(6.5, 5.5), dpi=300)

# --- 步骤 A: 绘制底层的“微弱云雾状”细胞散点 ---
for cat in categories:
    mask = adata_mp.obs[cluster_col] == cat
    ax.scatter(
        umap_coords[mask, 0], 
        umap_coords[mask, 1],
        c=color_dict[cat],
        s=12,             # 极小点径，呈现细腻质感
        alpha=0.50,       # 半透明，给前端线条留出对比度
        edgecolors='none',
        label=cat
    )

# --- 步骤 B: 绘制高亮黑实线 (PAGA 连线) ---
threshold = 0.05  # 连通性阈值，过滤掉微弱噪声连线
for i, cat1 in enumerate(categories):
    for j, cat2 in enumerate(categories):
        if i < j and paga_conn[i, j] > threshold:
            weight = paga_conn[i, j]
            # 线宽依据物理连通权重自动缩放
            linewidth = max(1.5, weight * 8.0) 
            x_vals = [centers[cat1][0], centers[cat2][0]]
            y_vals = [centers[cat1][1], centers[cat2][1]]
            ax.plot(
                x_vals, y_vals,
                color='black',
                linewidth=linewidth,
                zorder=4,               # 确保在散点之上
                solid_capstyle='round'  # 圆润线头
            )

# --- 步骤 C: 绘制节点实心圆圈与精致文字标签 ---
for i, cat in enumerate(categories):
    cx, cy = centers[cat]
    
    # 绘制高亮的节点圆实心点
    ax.scatter(
        cx, cy,
        c=color_dict[cat],
        s=180,              # 节点明显放大
        edgecolors='black', # 黑色描边
        linewidth=1.8,
        zorder=5            # 放在最上面
    )
    
    # 添加带白色极细外晕的加粗黑色字体（提高任何背景下的文字可读性）
    ax.text(
        cx, cy - 0.35,      # 稍微放于圆点下方一点位置
        cat,
        fontsize=11,
        fontweight='bold',
        color='black',
        ha='center',
        va='top',
        zorder=6,
        path_effects=[pe.withStroke(linewidth=2.5, foreground='white')]
    )

# 3. 最终排版美化 (去坐标轴，添加标题)
ax.set_title("Directed Trajectory of Macrophage Evolution", fontsize=13, fontweight='bold', pad=15)
ax.set_xticks([])
ax.set_yticks([])
ax.set_xlabel('')
ax.set_ylabel('')
# 去除四身边框 (完美吻合参考图风格)
for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.savefig('results/trajectory/Macrophage_PAGA_UMAP_Optimized.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================================
# 优化 1: 拟时间 vs 实际时间点 (使用您的 Time Point 主色调)
# =========================================================
plt.figure(figsize=(6, 5), dpi=150)

# 尝试从 adata 提取您预设的时间点颜色，如果没有则使用优雅的渐变蓝(Blues)
try:
    time_cats = adata_mp.obs[time_col].cat.categories
    time_colors = adata_mp.uns[f'{time_col}_colors']
    time_palette = dict(zip(time_cats, time_colors))
except KeyError:
    # 如果 adata 里没存时间点的颜色，默认使用类似您参考图的渐变蓝色系
    time_palette = 'Blues' 

sns.violinplot(
    data=adata_mp.obs, 
    x=time_col, 
    y='dpt_pseudotime', 
    palette=time_palette,  # <-- 关键修改：使用预设色或协调渐变色
    inner='quartile', 
    linewidth=1.5,
    scale='width'          # 让所有小提琴等宽，更美观
)
plt.title('Pseudotime Progression Across Real Time Points', fontsize=14, fontweight='bold')
plt.ylabel('Inferred Pseudotime', fontsize=12)
plt.xlabel('Actual Time Point', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# 去除顶部和右侧边框，贴合顶级期刊风格
sns.despine()
plt.tight_layout()
plt.savefig('results/trajectory/Pseudotime_vs_Realtime_Violin_ColorMatched.pdf', bbox_inches='tight')
plt.show()


# =========================================================
# 优化 2: 拟时间 vs 细胞亚群 (完美复刻您参考图左下角的风格)
# =========================================================
plt.figure(figsize=(4, 5), dpi=150)

# 精确提取您 UMAP 上的亚群主色调 (深蓝、浅蓝、粉色等)
cluster_cats = adata_mp.obs[cluster_col].cat.categories
cluster_colors = adata_mp.uns[f'{cluster_col}_colors']
cluster_palette = dict(zip(cluster_cats, cluster_colors))

# 按照拟时序中位数对亚群进行自动排序，让图表呈阶梯状上升
order_cats = adata_mp.obs.groupby(cluster_col)['dpt_pseudotime'].median().sort_values().index

sns.violinplot(
    data=adata_mp.obs, 
    x=cluster_col, 
    y='dpt_pseudotime', 
    order=order_cats,         # 自动按发育先后排序
    palette=cluster_palette,  # <-- 完全同步 UMAP 的亚群颜色
    inner='quartile', 
    linewidth=1.5
)
plt.title('Pseudotime across Macrophage Subtypes', fontsize=13, fontweight='bold', pad=15)
plt.ylabel('Pseudotime', fontsize=12)
plt.xlabel('')
plt.xticks(rotation=45, ha='right') # 标签倾斜防重叠
plt.grid(axis='y', linestyle='--', alpha=0.5)

sns.despine()
plt.tight_layout()
plt.savefig('results/trajectory/Pseudotime_vs_CellType_Violin.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
from scipy.sparse import issparse
import os

# 创建保存结果的目录
os.makedirs('results/trajectory', exist_ok=True)

# ================= 1. 安全获取表达矩阵与基因列表 =================
# 使用 adata_mp 提取数据
if adata_mp.raw is not None:
    print("检测到 raw 矩阵，将使用 raw 数据计算相关性...")
    expr_matrix = adata_mp.raw.X
    genes = list(adata_mp.raw.var_names)
else:
    print("未检测到 raw 矩阵，将使用当前 adata_mp.X 矩阵...")
    expr_matrix = adata_mp.X
    genes = list(adata_mp.var_names)

# 获取伪时间
pseudotime = adata_mp.obs['dpt_pseudotime'].values

# 排除伪时间为 NaN 的细胞
valid_idx = ~np.isnan(pseudotime)
pseudotime = pseudotime[valid_idx]
expr_matrix = expr_matrix[valid_idx, :]

if issparse(expr_matrix):
    expr_matrix = expr_matrix.tocsc()

# ================= 2. 预筛低表达基因 =================
if issparse(expr_matrix):
    mean_expression = np.array(expr_matrix.mean(axis=0)).flatten()
else:
    mean_expression = np.array(expr_matrix.mean(axis=0))

# 过滤：只计算平均表达量 > 0.05 的基因，极大缩减无意义计算
valid_gene_mask = mean_expression > 0.05
genes_to_test = [genes[i] for i, is_valid in enumerate(valid_gene_mask) if is_valid]
gene_indices = [i for i, is_valid in enumerate(valid_gene_mask) if is_valid]

print(f"原始基因数: {len(genes)} | 过滤后参与计算的基因数: {len(genes_to_test)}")

# ================= 3. 循环计算 Spearman 相关系数 =================
print("正在计算基因与拟时序的相关性 (Spearman)...")
correlation_results = []

for idx, gene_name in zip(gene_indices, genes_to_test):
    if issparse(expr_matrix):
        gene_expr = expr_matrix[:, idx].toarray().flatten()
    else:
        gene_expr = expr_matrix[:, idx]
        
    corr, pval = spearmanr(pseudotime, gene_expr)
    if not np.isnan(corr):
        correlation_results.append({
            'Gene': gene_name,
            'Correlation': corr,
            'P_value': pval
        })

corr_df = pd.DataFrame(correlation_results)

# ================= 4. FDR 多重检验校正 =================
if not corr_df.empty:
    corr_df['Adjusted_P_value'] = multipletests(corr_df['P_value'], method='fdr_bh')[1]
    
    threshold_corr = 0.25
    sig_df = corr_df[corr_df['Adjusted_P_value'] < 0.05]

    # 正相关极高 -> 晚期/持续上调基因
    up_genes_df = sig_df[sig_df['Correlation'] >= threshold_corr].sort_values(by='Correlation', ascending=False)
    # 负相关极高 -> 早期/持续下调基因
    down_genes_df = sig_df[sig_df['Correlation'] <= -threshold_corr].sort_values(by='Correlation', ascending=True)

    print(f"\n📊 筛选结果：")
    print(f"📈 显著【上调/晚期】基因: {len(up_genes_df)} 个")
    print(f"📉 显著【下调/早期】基因: {len(down_genes_df)} 个")

    # 导出 Excel
    output_filename = 'results/trajectory/Pseudotime_Dynamic_Genes.xlsx'
    with pd.ExcelWriter(output_filename) as writer:
        up_genes_df.to_excel(writer, sheet_name='Up_regulated_Genes', index=False)
        down_genes_df.to_excel(writer, sheet_name='Down_regulated_Genes', index=False)
    print(f"🎉 Excel 结果已保存至 '{output_filename}'")
else:
    print("❌ 未计算出任何有效基因，请检查 dpt_pseudotime 列是否存在。")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# =========================================================
# 1. 自动提取 Top 6 上调基因 和 Top 6 下调基因
# =========================================================
top_up = up_genes_df['Gene'].head(6).tolist()
top_down = down_genes_df['Gene'].head(6).tolist()
genes_to_plot = top_up + top_down

# 确保提取出的基因存在于当前的表达矩阵中
genes_to_plot = [g for g in genes_to_plot if g in genes]

# 准备绘图数据框
pt_df = pd.DataFrame({'Pseudotime': pseudotime})

for gene in genes_to_plot:
    gene_idx = genes.index(gene)
    if issparse(expr_matrix):
        pt_df[gene] = expr_matrix[:, gene_idx].toarray().flatten()
    else:
        pt_df[gene] = expr_matrix[:, gene_idx]

# =========================================================
# 2. 绘制 2行 6列 的面板图 (横向排版，第一排上调，第二排下调)
# =========================================================
# 18英寸宽，5.5英寸高，适合2x6的宽屏排版
fig, axs = plt.subplots(2, 6, figsize=(18, 5.5), dpi=200)
axs = axs.flatten()

for i, gene in enumerate(genes_to_plot):
    ax = axs[i]
    # 前6个为上调基因 (红色趋势线)，后6个为下调基因 (绿色趋势线)
    is_up = i < 6
    line_color = 'red' if is_up else 'mediumseagreen'
    
    # 绘制拟合趋势曲线
    sns.regplot(
        data=pt_df, 
        x='Pseudotime', 
        y=gene, 
        scatter=False, 
        order=3, 
        color=line_color,
        line_kws={'linewidth': 3},
        ax=ax
    )
    
    # 样式微调
    ax.set_title(gene, fontsize=12, fontweight='bold', fontstyle='italic')
    
    # 只有第二排（下调基因，索引 >= 6）显示 X 轴标签 "Pseudotime"
    ax.set_xlabel('Pseudotime' if i >= 6 else '', fontsize=10)
    ax.set_ylabel('Expression' if i % 6 == 0 else '', fontsize=10) # 只在每排的第一列显示Y轴刻度标签
    
    # 限制 X 轴范围在 0 到 1 之间
    ax.set_xlim(-0.02, 1.02)
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # 添加淡淡的浅蓝色底色背景区
    ax.axvspan(0, 1, color='#E6F0FA', alpha=0.25, zorder=0)

# 如果基因数量不足 12 个，关闭多余的子图
if len(genes_to_plot) < 12:
    for j in range(len(genes_to_plot), 12):
        fig.delaxes(axs[j])

plt.tight_layout()
# 保存为适合排版的高清 PDF
plt.savefig('results/trajectory/Adata_mp_Top6_Row_Trends.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from scipy.sparse import issparse
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

# ================= 1. 提取所有差异基因并合并 =================
down_genes = down_genes_df['Gene'].tolist()
up_genes = up_genes_df['Gene'].tolist()
all_heatmap_genes = down_genes + up_genes

# 按照拟时序从小到大对细胞进行排序
sorted_indices = np.argsort(pseudotime)

# 提取表达矩阵数据
heatmap_expr = []
for gene in all_heatmap_genes:
    gene_idx = genes.index(gene)
    if issparse(expr_matrix):
        val = expr_matrix[sorted_indices, gene_idx].toarray().flatten()
    else:
        val = expr_matrix[sorted_indices, gene_idx]
    heatmap_expr.append(val)

# 构建初始的 DataFrame
df_heatmap = pd.DataFrame(heatmap_expr, index=all_heatmap_genes)

# ================= 2. 表达量平滑与安全过滤 =================
window = max(int(len(sorted_indices) * 0.05), 10)
df_smoothed = df_heatmap.T.rolling(window=window, center=True, min_periods=1).mean().T

std_devs = df_smoothed.std(axis=1)
valid_genes_mask = std_devs > 0
df_smoothed = df_smoothed.loc[valid_genes_mask]

filtered_down_genes = [g for g in down_genes if g in df_smoothed.index]
filtered_up_genes = [g for g in up_genes if g in df_smoothed.index]
filtered_all_genes = filtered_down_genes + filtered_up_genes

df_smoothed = df_smoothed.loc[filtered_all_genes]

# ================= 3. 标准 2D NumPy 矩阵进行 Z-score =================
matrix_data = df_smoothed.to_numpy(dtype=np.float64)
matrix_norm = zscore(matrix_data, axis=1)
matrix_norm = np.nan_to_num(matrix_norm, nan=0.0, posinf=0.0, neginf=0.0)
df_norm = pd.DataFrame(matrix_norm, index=filtered_all_genes)

# ================= 4. 绘制热图与基因分组侧边栏 =================
# 调整画布高度为 12，给基因名留出更多垂直空间
fig, (ax_bar, ax_heat) = plt.subplots(
    1, 2, 
    figsize=(10, 12), 
    dpi=200, 
    gridspec_kw={'width_ratios': [0.25, 10], 'wspace': 0.01} # wspace极小，使侧边条紧贴热图
)

# 4.1 绘制左侧的分组颜色条
group_tags = np.array([0] * len(filtered_down_genes) + [1] * len(filtered_up_genes)).reshape(-1, 1)
cmap_bar = sns.color_palette(['mediumseagreen', 'red'])

sns.heatmap(
    group_tags,
    cmap=cmap_bar,
    cbar=False,
    xticklabels=False,
    yticklabels=False,
    ax=ax_bar
)

# 4.2 绘制右侧的主热图
sns.heatmap(
    df_norm,
    cmap='RdBu_r',       
    vmin=-2, vmax=2,     
    xticklabels=False,   
    yticklabels=True,    
    cbar_kws={
        'label': 'Z-score Normalized Expression',
        'shrink': 0.35,   # 显著缩小色条高度
        'aspect': 18,     # 让色条更修长
        'pad': 0.08       # 往右偏移，防止与右侧的基因名重叠
    },
    ax=ax_heat
)

# 【核心优化】将基因名移动到热图右侧，避免与左侧颜色条冲突
ax_heat.yaxis.tick_right()
ax_heat.yaxis.set_label_position("right")

# 美化基因名刻度样式
ax_heat.tick_params(axis='y', labelsize=5.5, rotation=0, right=True)
for label in ax_heat.get_yticklabels():
    label.set_style('italic')

# 设置轴标签与标题
ax_heat.set_xlabel('Pseudotime Progression', fontsize=12, fontweight='bold', labelpad=10)
ax_heat.set_title(
    f'Pseudotime-Dependent Genes (Down: {len(filtered_down_genes)} | Up: {len(filtered_up_genes)})', 
    fontsize=14, 
    fontweight='bold', 
    pad=35  # 增加间距给上方的图例留出空间
)

# 【核心优化】创建顶部并排的图例，避免遮挡
legend_elements = [
    Patch(facecolor='mediumseagreen', label=f'Down-regulated / Early ({len(filtered_down_genes)})'),
    Patch(facecolor='red', label=f'Up-regulated / Late ({len(filtered_up_genes)})')
]
ax_heat.legend(
    handles=legend_elements, 
    loc='lower left', 
    bbox_to_anchor=(0.0, 1.01),  # 定位到热图正上方
    ncol=2,                      # 水平排列
    frameon=False,               # 去掉边框更清爽
    fontsize=11
)

plt.tight_layout()
# 保存为适合排版的高清 PDF
plt.savefig('results/trajectory/Adata_mp_All_Dynamic_Heatmap_optimized.pdf', bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import issparse
import numpy as np
import pandas as pd

# =========================================================
# 1. 提取降维坐标
# =========================================================
if 'X_umap' in adata_mp.obsm: 
    coords = adata_mp.obsm['X_umap'] 
elif 'X_tsne' in adata_mp.obsm: 
    coords = adata_mp.obsm['X_tsne'] 
else: 
    coords = adata_mp.obsm['X_draw_graph_fa'] if 'X_draw_graph_fa' in adata_mp.obsm else adata_mp.obsm[list(adata_mp.obsm.keys())[0]] 

x_coords = coords[:, 0] 
y_coords = coords[:, 1] 

# =========================================================
# 2. 指定统一的基因列表（与拟时序图完全一致）
# =========================================================
use_raw = adata_mp.raw is not None
gene_pool = list(adata_mp.raw.var_names) if use_raw else list(adata_mp.var_names) 

# 与第二张拟时序图完全统一的基因列表
desired_up = ['CD83', 'NAMPT', 'HIF1A', 'FTH1', 'ZNF331', 'SRGN']
desired_down = ['RIPOR2', 'CD300E', 'FGR', 'SLC11A1', 'FRY', 'FCN1']

# 检查基因是否存在
missing_up = set(desired_up) - set(gene_pool)
missing_down = set(desired_down) - set(gene_pool)

if missing_up:
    print(f"⚠️ 警告: 上调基因缺失: {missing_up}")
if missing_down:
    print(f"⚠️ 警告: 下调基因缺失: {missing_down}")

# 合并绘制清单
genes_to_plot = desired_up + desired_down

print("【最终绘制基因清单】") 
print(f"第一行 (Up/Red):   {desired_up}") 
print(f"第二行 (Down/Blue): {desired_down}") 

# =========================================================
# 3. 绘制 2行 6列 布局
# =========================================================
fig, axs = plt.subplots(2, 6, figsize=(16, 5), dpi=300)
axs = axs.flatten() 

for i, gene in enumerate(genes_to_plot): 
    ax = axs[i] 
    
    # 容错处理：如果基因不在数据集中则跳过
    if gene not in gene_pool:
        ax.axis('off')
        ax.set_title(f"{gene}\n(Not Found)", fontsize=10, color='gray')
        continue
    
    # 获取表达量 (自动识别从 raw 还是 X 中提取) 
    if use_raw: 
        gene_idx = gene_pool.index(gene) 
        expr_matrix = adata_mp.raw.X
    else: 
        gene_idx = gene_pool.index(gene) 
        expr_matrix = adata_mp.X
        
    if issparse(expr_matrix): 
        expr = expr_matrix[:, gene_idx].toarray().flatten() 
    else: 
        expr = expr_matrix[:, gene_idx] 
    
    # 前 6 个为上调 (红色)，后 6 个为下调 (蓝色) 
    is_up = i < 6
    cmap_color = 'Reds' if is_up else 'Blues' 
    
    # 绘制背景灰色散点
    ax.scatter(x_coords, y_coords, c='lightgrey', s=1.2, alpha=0.3, edgecolors='none', zorder=1) 
    
    # 按照表达量排序绘制，让高表达细胞置于最上层
    order = np.argsort(expr) 
    sc = ax.scatter( 
        x_coords[order], 
        y_coords[order], 
        c=expr[order], 
        cmap=cmap_color, 
        s=2.2, 
        alpha=0.9, 
        edgecolors='none', 
        zorder=2
    ) 
    
    # 标题样式 (加粗斜体) 
    ax.set_title(gene, fontsize=13, fontweight='bold', fontstyle='italic', pad=8) 
    
    # 移除坐标轴和边框
    ax.set_xticks([]) 
    ax.set_yticks([]) 
    for spine in ax.spines.values(): 
        spine.set_visible(False) 
        
    # 添加 Colorbar
    cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04) 
    cbar.ax.tick_params(labelsize=8) 
    cbar.outline.set_visible(False) 

plt.tight_layout() 
plt.savefig('results/trajectory/Adata_mp_6Up_6Down_Expression.pdf', bbox_inches='tight') 
plt.show()

In [ ]:
import pandas as pd
import gseapy as gp
from gseapy import barplot
import matplotlib.pyplot as plt
import os

# 创建保存富集结果的文件夹
os.makedirs('results/enrichment', exist_ok=True)

up_genes = up_genes_df['Gene'].dropna().unique().tolist()
down_genes = down_genes_df['Gene'].dropna().unique().tolist()

go_sets = [
    'GO_Biological_Process_2023',
    'GO_Cellular_Component_2023',
    'GO_Molecular_Function_2023'
]
organism = 'human'  # 如果是小鼠请改为 'mouse'

# =========================================================
# 1. 上调基因 GO 富集分析与绘图
# =========================================================
print("\n正在运行上调基因 GO 富集分析...")
try:
    enr_up = gp.enrichr(
        gene_list=up_genes,
        gene_sets=go_sets,
        organism=organism,
        outdir=None,
        cutoff=0.05
    )
    
    # 筛选出 Biological Process
    go_bp_up = enr_up.results[enr_up.results['Gene_set'] == 'GO_Biological_Process_2023'].copy()
    
    if not go_bp_up.empty:
        # 【强力清洗】：通过分割字符串直接扔掉 (GO: 及其后面的内容
        go_bp_up['Term'] = go_bp_up['Term'].apply(lambda x: x.split(' (GO:')[0].strip())
        
        plt.figure(figsize=(8, 5))
        barplot(go_bp_up, column="Adjusted P-value", top_term=10, size=10, color='#e28743')
        plt.title("Up-regulated Genes: GO Biological Process", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig('results/enrichment/Up_genes_GO_BP_barplot.pdf', bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f"上调基因 GO 富集运行失败: {e}")

# =========================================================
# 2. 下调基因 GO 富集分析与绘图
# =========================================================
print("\n正在运行下调基因 GO 富集分析...")
try:
    enr_down = gp.enrichr(
        gene_list=down_genes,
        gene_sets=go_sets,
        organism=organism,
        outdir=None,
        cutoff=0.05
    )
    
    # 筛选出 Biological Process
    go_bp_down = enr_down.results[enr_down.results['Gene_set'] == 'GO_Biological_Process_2023'].copy()
    
    if not go_bp_down.empty:
        # 【强力清洗】：通过分割字符串直接扔掉 (GO: 及其后面的内容
        go_bp_down['Term'] = go_bp_down['Term'].apply(lambda x: x.split(' (GO:')[0].strip())
        
        plt.figure(figsize=(8, 5))
        barplot(go_bp_down, column="Adjusted P-value", top_term=10, size=10, color='#336699')
        plt.title("Down-regulated Genes: GO Biological Process", fontsize=12, fontweight='bold')
        plt.tight_layout()
        plt.savefig('results/enrichment/Down_genes_GO_BP_barplot.pdf', bbox_inches='tight')
        plt.show()
except Exception as e:
    print(f"下调基因 GO 富集运行失败: {e}")
